# Regularization and Training Control

Goal: understand techniques that improve neural-network generalization and training stability.

Topics:
- dropout
- batch normalization
- learning-rate scheduling
- comparing regularized models

In [1]:
# tools imported

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader, random_split
import matplotlib.pyplot as plt

In [2]:
# noisy nonlinear dataset creation

torch.manual_seed(42)

x = torch.linspace(-3, 3, 120).reshape(-1, 1)

y = (
    0.5 * x**3
    - 2 * x**2
    + x
    + 3
)

y += 1.5 * torch.randn_like(y)

In [3]:
# training / validation split

dataset = TensorDataset(x, y)

train_dataset, val_dataset = random_split(
    dataset,
    [60, 60],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=10,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=10,
    shuffle=False
)

In [4]:
# set device

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", device)

Using: cuda


## Dropout Method

Dropout randomly disables some neuron outputs during training, with the idea of stopping the network from depending too much on certain neurons

In [6]:
# model definition

class DropoutNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(1, 256), # linear
            nn.ReLU(), # ReLU
            nn.Dropout(p=0.2), # dropout

            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Dropout(p=0.2),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(p=0.2),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x)

In [7]:
dropout_model = DropoutNetwork().to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    dropout_model.parameters(),
    lr=0.001
)